Install required libraries and import then

In [ ]:
# --- Step 1: Install libraries ---
!pip install noisereduce -q
!pip install soundfile -q  # <-- Audio save karne ke liye zaroori
print("✅ Libraries installed.")

# --- Step 2: Import everything ---
import pandas as pd
import numpy as np
import os
import librosa
from numpy import genfromtxt
from tensorflow.keras.models import load_model
import noisereduce as nr

import soundfile as sf
import IPython.display as ipd
import copy

print("✅ Sab kuch import ho gaya.")

Define all the path

In [ ]:
# --- Step 3: Saare paths yahaan define karein (Updated) ---
import pandas as pd
import os

# 1. Model ka path (Aapke bataye gaye dataset naam 'model-h5' ke aadhar par)
model_path = '/kaggle/input/model-h5/model.h5'

# 2. Test audio file ka path
filename = '/kaggle/input/audio-mp3/project explanation.mp3' # <-- Yahaan apni audio file ka naam check kar lein

# 3. UrbanSound8K Dataset ka Path
urbansound_path = '/kaggle/input/urbansound8k/'
urbansound_metadata_file = os.path.join(urbansound_path, 'UrbanSound8K.csv')

# 4. Output file ka path
output_filename = '/kaggle/working/clean.wav'

# --- Ab files check karein ---
print("File paths set. Checking...")

if not os.path.exists(model_path):
    print(f"❌ Error: Model file '{model_path}' nahi mili.")
    print("FIX: Kya aapne 'model-h5' dataset add kiya hai aur uske andar 'model.h5' file hai?")
else:
    print(f"✅ Model file mil gayi: {model_path}")

if not os.path.exists(filename):
    print(f"❌ Error: Audio file '{filename}' nahi mili.")
else:
    print(f"✅ Audio file mil gayi: {filename}")

if not os.path.exists(urbansound_metadata_file):
    print(f"❌ Error: UrbanSound8K metadata '{urbansound_metadata_file}' nahi mili.")
else:
    try:
        metadata_df = pd.read_csv(urbansound_metadata_file)
        print(f"✅ UrbanSound8K metadata file mil gayi aur load ho gayi hai.")
    except Exception as e:
        print(f"❌ Error: Metadata file load karne mein error: {e}")

Load the model and Diagonose the audio file

In [ ]:
# --- Step 4: Load Model ---
print("Model load kar raha hoon...")
try:
    model = load_model(model_path)
    print('\n\n\n ✅ Model Loaded \n\n\n')
    model.summary()
except Exception as e:
    print(f"❌ Error: Model load nahi ho paya. Error: {e}")
    print("FIX: Kya aapki model.h5 file ek Keras/TensorFlow model hai?")


# --- Step 5: FIXED DENOISE FUNCTION (Final Fix) ---
def denoise(data_clip, noise_file_path, sr):
    print(f"Reducing noise using profile: {noise_file_path}")
    try:
        # Load noise clip as mono
        noise_clip, sr_noise = librosa.load(noise_file_path, sr=sr, mono=True) 
        
        # --- FIX: 'verbose=False' argument hata diya gaya hai ---
        reduced_noise = nr.reduce_noise(y=data_clip,
                                        y_noise=noise_clip,
                                        sr=sr)
        # --- END FIX ---
        
        return reduced_noise
    except Exception as e:
        print(f"Error loading or processing noise file {noise_file_path}: {e}")
        return data_clip

print("✅ Denoise function tayyar hai (Final Update).")

Final Testing

In [ ]:
# Aapke model ke index (0-9) ko UrbanSound8K ke class names se map karna
urbansound_class_map = {
    0: 'air_conditioner',  # Aapka 'Windy'
    1: 'car_horn',         # Aapka 'Horn'
    2: 'children_playing', # Aapka 'Children-noise'
    3: 'dog_bark',
    4: 'drilling',
    5: 'engine_idling',
    6: 'gun_shot',
    7: 'jackhammer',
    8: 'siren',
    9: 'street_music'
}

try:
    # --- Step 6: Preprocessing (FIXED) ---
    print(f"\nLoading and preprocessing {filename}...")
    x_test = []
    
    # --- FIX: `mono=True` add kiya gaya hai ---
    y, sr = librosa.load(filename, sr=None, mono=True)
    print(f"Audio loaded (mono). Shape: {y.shape}, Sample Rate: {sr}, Duration: {len(y)/sr:.2f}s")
    # --- END FIX ---

    # 1. Extract MFCCs (Feature 1)
    mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40).T, axis=0)
    # 2. Extract Melspectrogram (Feature 2)
    melspectrogram = np.mean(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=40, fmax=8000).T, axis=0)

    # Stack karke (40, 2) shape banayein
    features = np.reshape(np.vstack((mfccs, melspectrogram)).T, (40, 2))
    
    x_test.append(features)
    x_test = np.array(x_test)
    x_test = np.reshape(x_test, (x_test.shape[0], 40, 2, 1))
    print('\nFinal test shape: ', x_test.shape)

    # --- Step 7: Prediction ---
    ans = model.predict(x_test)

    print('\nClass 0: Windy \n Class 1: Horn\n Class 2: Children-noise \n Class 3: Dog Bark \n Class 4: Drilling \n Class 5: Engine Idling\n Class 6: Gun Shot \n Class 7: Jackhammer\n Class 8: Siren \n Class 9: Street music\n')

    my_dict = {0: 'Windy', 1: 'Horn', 2: 'Children-noise', 3: 'Dog Bark', 4: 'Drilling', 5: 'Engine Idling', 6: 'Gun Shot', 7: 'Jackhammer', 8: 'Siren', 9: 'Street music'}

    # Prediction logic
    x = copy.copy(ans[0])
    x = list(x)
    arr = []
    ls = list(ans[0])

    while (len(x) > 8): # Top 2 predictions
        aud = max(x)
        index = ls.index(aud)
        x.remove(aud)
        arr.append(index)

    print('Resulted Index: ', arr)
    print('\nNoises Present: ')
    for idx in arr:
        print(my_dict[idx])

    # --- Step 8: Denoising (LOGIC FIX FOR URBANSOUND8K) ---
    print("\nStarting noise reduction...")
    cleaned_data = copy.copy(y)
    noise_files_to_use = []

    # Loop 1: Saari zaroori noise files ikattha karein
    for i in arr: # i = prediction index (e.g., 8)
        if i in urbansound_class_map:
            noise_class_name = urbansound_class_map[i]
            noise_file_row = metadata_df[metadata_df['class'] == noise_class_name].sample(1).iloc[0]
            file_name = noise_file_row['slice_file_name']
            fold_num = noise_file_row['fold']
            full_noise_path = os.path.join(urbansound_path, f"fold{fold_num}", file_name)
            
            if os.path.exists(full_noise_path):
                noise_files_to_use.append(full_noise_path)
            else:
                print(f"Warning: Dhoondi gayi file {full_noise_path} nahi mili. Skip kar raha hoon.")
        else:
            print(f'Index {i} ke liye noise profile defined nahi hai, skip kar raha hoon.')

    # Loop 2: Ek ke baad ek noise reduction apply karein
    print(f"\n{len(noise_files_to_use)} noise profiles dhoondh liye gaye hain. Ab apply kar raha hoon...")
    for noise_file in noise_files_to_use:
        if os.path.exists(noise_file):
            cleaned_data = denoise(cleaned_data, noise_file, sr)
        else:
            print(f"Warning: Noise file {noise_file} nahi mili. Skip kar raha hoon.")

    # --- Step 9: Final result save aur play karein ---
    sf.write(output_filename, cleaned_data, sr)
    print(f"\nSab noise reduction poora. Cleaned Audio save ho gaya hai: {output_filename}")

    # --- "HEAR" BUTTON ---
    print("\n--- Original Audio ---")
    ipd.display(ipd.Audio(y, rate=sr))
    print("\n--- Cleaned Audio ---")
    ipd.display(ipd.Audio(cleaned_data, rate=sr))

    # --- "DOWNLOAD" BUTTON (Kaggle ka tareeqa) ---
    print(f"\n'{output_filename}' ko download karne ke liye:")
    print("1. Is notebook ke upar 'Save Version' button dabayein.")
    print("2. Jab 'Save Version' poora ho jaaye, notebook viewer par jaayein.")
    print("3. 'Output' tab par click karein.")
    print("4. Aapko 'clean.wav' file mil jayegi.")


except FileNotFoundError as e:
    print(f"❌ FATAL ERROR: File nahi mili.")
    print(e)
    print("Kripya Cell 3 mein file path check karein.")
except Exception as e:
    print(f"Ek unexpected error aaya: {e}")